# Общая схема прогнозирования перцентилей Avito

В исследовании сопоставлены одна модель на все три режима, модели по режимам,
модели по ячейкам внутри режима и гибрид. Дополнительные ориентиры — эмпирические
квартили и вручную заданное соответствие. Объект прогноза — опубликованная
точная сумма или середина закрытой вилки. Прогноз всегда относится к указанным
единице и налоговой базе.

Сначала можно выполнить ноутбук на готовых пакетах. Ячейка с настройкой входного CSV
и правил маршрутизации позволяет проверить любое число своих вакансий.
[Сводка и ограничения](experiments/2026-11-18_salary_routing/README.md) ·
[Пакеты и схема входа](models/2026-11-18_salary_routing/README.md)

In [1]:
import copy
import json
import sys
from pathlib import Path

import pandas as pd
from IPython.display import display

cwd = Path.cwd().resolve()
root = next(p for p in (cwd, *cwd.parents) if (p / 'ML/salary_routing.py').is_file())
if str(root / 'ML') not in sys.path:
    sys.path.insert(0, str(root / 'ML'))

import salary_routing as routing

exp = routing.EXPERIMENT_DIR
models = routing.MODEL_DIR
catalog = routing.load_catalog(models)['models']

def read(name):
    return pd.read_csv(exp / (name + '.csv'), dtype={'id': str})

rule_file = exp / 'routing_policy.json'
policy = json.loads(rule_file.read_text(encoding='utf-8'))
display(pd.DataFrame.from_dict(catalog, orient='index')
        .rename_axis('approach')[['method','scope','fit_points','artifact']])


,method,scope,fit_points,artifact
approach,,,,
global,catboost,{},1720,global.joblib
empirical,empirical,{},1720,empirical.joblib
mode_ordinary,catboost,{'mode': 'ordinary'},1272,mode_ordinary.joblib
mode_watch,catboost,{'mode': 'watch'},359,mode_watch.joblib
mode_remote,catboost,{'mode': 'remote'},89,mode_remote.joblib
segment_17fa6ea3bf72,catboost,"{'mode': 'ordinary', 'unit': 'shift', 'tax': '...",38,segment_17fa6ea3bf72.joblib
segment_624001cf5ad1,catboost,"{'mode': 'ordinary', 'unit': 'shift', 'tax': '...",35,segment_624001cf5ad1.joblib
segment_cb1c79cab67f,catboost,"{'mode': 'ordinary', 'unit': 'shift', 'tax': '...",48,segment_cb1c79cab67f.joblib
segment_5e81ccbe361e,catboost,"{'mode': 'ordinary', 'unit': 'shift', 'tax': '...",67,segment_5e81ccbe361e.joblib


## Задать подход и получить прогноз

Поменяйте input_path для своего CSV. Требуются mode, unit, tax. Для точной ячейки
используются city_resolved и profession_resolved. Допустимы дополнительные поля
карточки. Поле salary_raw и сама сумма для прогноза не нужны. Точные условия из
rules проверяются сверху вниз. Затем используется default. Если пакет для
выбранного правила неприменим, проверяются подходы из fallback. Значение
abstain останавливает маршрут и оставляет прогноз пустым.

Для проверки новых соответствий сначала правьте копию правил в этой ячейке.
После проверки можно сохранить правила вручную в отдельный JSON и указать
новый rule_file. Новые модели появляются в списке после явного добавления
совместимого пакета и записи о его области в catalog.json.

In [2]:
input_path = exp / 'example_vacancies.csv'
vacancies = pd.read_csv(input_path, dtype=str, keep_default_na=False)

# Пример ручной правки, если нужен отдельный вариант.
# policy['rules'][0]['approach'] = 'mode_ordinary'
# policy['fallback'] = ['empirical']
routed = routing.predict_vacancies(vacancies, policy, models)
fields = ['id','segment','rule','approach','p25','p50','p75','reference_or_reason','limitations']
display(routed[fields].round(1))


,id,segment,rule,approach,p25,p50,p75,reference_or_reason,limitations
0,2971866871,ordinary | shift | net | Кассир | Санкт-Петербург,spb_cashier_statistic,empirical,3375.0,3650.0,4000.0,city_profession,published_offers; mode_evidence:residual
1,1012505611,ordinary | shift | net | Продавец | Санкт-Пете...,spb_seller_statistic,empirical,3750.0,3850.0,5500.0,city_profession,published_offers; mode_evidence:residual
2,4347741400,ordinary | shift | net | Комплектовщик | Санкт...,spb_picker_statistic,empirical,4014.0,4625.0,5500.0,city_profession,published_offers; mode_evidence:residual
3,1317059545,watch | shift | net | | Щёлково,default,global,3798.7,4489.6,5728.0,conditional_model,published_offers; missing_segment_fields; veri...
4,2459477803,remote | shift | net | | Красноярск,default,global,2675.8,3025.6,4451.1,conditional_model,published_offers; missing_segment_fields
5,2526217995,remote | shift | net | | Тольятти,default,global,4289.2,5522.2,8164.6,conditional_model,published_offers; missing_segment_fields; mode...
6,1040408058,ordinary | unspecified | unknown | Водитель гр...,default,abstain,NaN,NaN,NaN,unsupported_or_unknown_unit,unsupported_or_unknown_unit
7,1172455832,ordinary | shift | unknown | Продавец | Екатер...,default,abstain,NaN,NaN,NaN,unknown_tax,unknown_tax
8,1771751639,ordinary | hour | net | Грузчик | Пенза,default,global,505.9,619.5,708.8,conditional_model,published_offers; mode_evidence:residual
9,4288354022,ordinary | shift | net | Грузчик | Москва,default,global,3726.5,4197.6,5592.1,conditional_model,published_offers; mode_evidence:residual


## Контроль ручной замены

На демонстрационных строках показано, как изменить решение для одной
сопоставимой ячейки. Правка меняет маршрут к уже сохранённому пакету.
Исходный JSON остаётся шаблоном для следующих проверок.

In [3]:
manual = copy.deepcopy(policy)
fields_check = ['id','segment','approach','p25','p50','p75','limitations']
# Выбираем первую показанную ячейку с готовым сегментным пакетом.
defined = [(name, item['scope']) for name, item in catalog.items() if name.startswith('segment_')
           and routing.match(vacancies, item['scope']).any()]
if defined:
    specialist, filters = defined[0]
    selected = routing.match(vacancies, filters)
    manual['rules'].insert(0, dict(id='review_specialist', when=filters, approach=specialist,
                                   why='Пример ручной замены для проверки поведения',
                                   evidence='cell_comparison.csv'))
    updated = routing.predict_vacancies(vacancies, manual, models)
    display(pd.concat([routed.loc[selected, fields_check].assign(version='base'),
                       updated.loc[selected, fields_check].assign(version='edited')]).round(1))


,id,segment,approach,p25,p50,p75,limitations,version
9,4288354022,ordinary | shift | net | Грузчик | Москва,global,3726.5,4197.6,5592.1,published_offers; mode_evidence:residual,base
9,4288354022,ordinary | shift | net | Грузчик | Москва,segment_17fa6ea3bf72,3915.8,4297.3,4937.3,published_offers; mode_evidence:residual,edited


## Сопоставимая оценка схем

Три тестовых фолда сформированы по группам работодателей и найденных дублей.
Каждый тестовый ID используется одной моделью каждого варианта. Медиана для
масштабирования ошибки рассчитана по единице и налоговой базе в обучающей части.
Показатель scaled_pinball безразмерный. Pinball и MAE в рублях приводятся отдельно
по единице и базе. Покрытие P25–P75 относится к вероятностному интервалу
и имеет ориентир 50%. Охват маршрута обозначает долю вакансий, получивших прогноз.

Колонка available показывает качество на тех строках, где схема сработала.
common_four сравнивает четыре основные схемы на одинаковых строках.
common_broad дополнительно сравнивает широкие варианты на их общем пересечении.
Эти срезы публикуются вместе, поскольку локальные модели видят меньший пул.

In [4]:
summary = read('summary')
quality = read('quality')
native = read('quality_by_unit')
cols = ['scope','scheme','total','contextual','eligible','scored','scored_eligible',
        'route_coverage','eligible_coverage','n','scaled_pinball','coverage']
display(summary.loc[summary.scope.eq('all'), cols].round(3))
display(summary.loc[summary.scope.isin(['ordinary','watch','remote']), cols].round(3))
display(quality.loc[quality.scope.eq('all') & quality['sample'].ne('available'),
                    ['sample','scheme','n','scaled_pinball','scaled_p50_mae','coverage']].round(3))
display(native.query("sample == 'available' and unit == 'shift' and tax == 'net' and scope in ['ordinary','watch','remote','remote_explicit']")
        [['scope','scheme','n','pinball','mae','medae','coverage','width']].round(2))


,scope,scheme,total,contextual,eligible,scored,scored_eligible,route_coverage,eligible_coverage,n,scaled_pinball,coverage
0,all,by_mode,31674,6398,3170,6382,3160,0.201,0.997,3160,0.843,0.460
1,all,by_segment,31674,6398,3170,2308,1250,0.073,0.394,1250,0.185,0.298
2,all,empirical,31674,6398,3170,6382,3160,0.201,0.997,3160,0.769,0.481
3,all,global,31674,6398,3170,6382,3160,0.201,0.997,3160,0.810,0.477
4,all,hybrid,31674,6398,3170,6382,3160,0.201,0.997,3160,0.813,0.388
5,all,manual,31674,6398,3170,6382,3160,0.201,0.997,3160,0.810,0.477


,scope,scheme,total,contextual,eligible,scored,scored_eligible,route_coverage,eligible_coverage,n,scaled_pinball,coverage
12,ordinary,by_mode,19649,4806,2368,4806,2368,0.245,1.000,2368,0.253,0.483
13,ordinary,by_segment,19649,4806,2368,2161,1174,0.110,0.496,1174,0.177,0.292
14,ordinary,empirical,19649,4806,2368,4806,2368,0.245,1.000,2368,0.259,0.480
15,ordinary,global,19649,4806,2368,4806,2368,0.245,1.000,2368,0.256,0.474
16,ordinary,hybrid,19649,4806,2368,4806,2368,0.245,1.000,2368,0.261,0.363
17,ordinary,manual,19649,4806,2368,4806,2368,0.245,1.000,2368,0.256,0.475
18,remote,by_mode,4833,243,150,228,141,0.047,0.940,141,0.590,0.560
19,remote,by_segment,4833,243,150,0,0,0.000,0.000,0,NaN,NaN
20,remote,empirical,4833,243,150,228,141,0.047,0.940,141,0.526,0.539
21,remote,global,4833,243,150,228,141,0.047,0.940,141,0.505,0.468


,sample,scheme,n,scaled_pinball,scaled_p50_mae,coverage
7,common_four,global,1250,0.176,0.374,0.521
12,common_broad,global,3160,0.810,1.778,0.477
26,common_four,by_mode,1250,0.179,0.386,0.505
31,common_broad,by_mode,3160,0.843,1.781,0.460
45,common_four,by_segment,1250,0.185,0.396,0.298
57,common_four,hybrid,1250,0.185,0.396,0.298
62,common_broad,hybrid,3160,0.813,1.787,0.388
76,common_four,empirical,1250,0.179,0.389,0.482
81,common_broad,empirical,3160,0.769,1.683,0.481
95,common_four,manual,1250,0.177,0.375,0.524


,scope,scheme,n,pinball,mae,medae,coverage,width
7,ordinary,global,2281,1270.49,2680.95,866.24,0.48,1856.09
10,watch,global,651,14716.52,32811.07,1110.42,0.49,2418.01
12,remote,global,122,1730.78,3640.77,1000.09,0.45,2079.84
15,remote_explicit,global,58,2152.77,4439.88,881.90,0.57,1815.08
45,ordinary,by_mode,2281,1247.71,2657.73,916.61,0.49,1978.50
48,watch,by_mode,651,15539.94,32804.53,1323.61,0.36,2049.43
50,remote,by_mode,122,1996.96,4047.40,1417.50,0.59,4139.06
53,remote_explicit,by_mode,58,2223.96,4382.50,807.77,0.67,3660.52
83,ordinary,by_segment,1174,895.15,1917.28,971.68,0.29,841.89
86,watch,by_segment,76,1628.95,3342.31,562.91,0.38,1365.59


## Локальные ячейки, ошибки и запасной маршрут

Локальные пакеты допускались только при достаточной опоре обучения и валидации.
Наличие пакета означает доступность исследования по ячейке. Для ручного
назначения применяйте парные результаты и новые независимые данные.

На вахте отдельно показана чувствительность к большим значениям, помеченным
в карточках как сменные. Сравнение на хвосте выполняется после оценки.
Для удалёнки показан разрез, подтверждённый явным полем карточки.
При неизвестном периоде и налоговой базе схема возвращает прозрачную причину
отказа; пользователь может самостоятельно задать подтверждённый контекст.


In [5]:
local = read('cell_comparison')
keys = ['mode','unit','tax','profession_resolved','city_resolved']
local_pivot = local.pivot(index=keys, columns='scheme', values='pinball').reset_index()
size = local.loc[local.scheme.eq('by_segment'), keys + ['n']]
display(size.merge(local_pivot,on=keys).sort_values('n',ascending=False).head(15).round(1))
display(read('paired_vs_global').query("scope in ['all','watch','watch_shift_le_50000','remote_explicit']").round(3))
display(read('largest_errors').query("scheme == 'global' and unit == 'shift'")
        .groupby('mode').head(3)[['mode','id','y','p50','abs_error_p50']].round(1))


,mode,unit,tax,profession_resolved,city_resolved,n,by_mode,by_segment,empirical,global,hybrid,manual
9,ordinary,shift,net,Курьер,Санкт-Петербург,212,564.9,685.7,615.1,546.3,685.7,546.3
7,ordinary,shift,net,Курьер,Москва,205,670.9,700.9,685.2,676.4,700.9,676.4
5,ordinary,shift,net,Курьер,Екатеринбург,143,633.1,702.0,597.0,619.4,702.0,619.4
6,ordinary,shift,net,Курьер,Краснодар,125,594.4,614.1,606.2,621.7,614.1,621.7
11,ordinary,shift,net,Продавец,Санкт-Петербург,107,2225.3,2196.6,2206.0,2163.1,2196.6,2206.0
4,ordinary,shift,net,Комплектовщик,Санкт-Петербург,97,889.9,905.0,881.1,870.2,905.0,881.1
8,ordinary,shift,net,Курьер,Рязань,79,528.6,493.1,499.9,481.8,493.1,481.8
12,watch,shift,net,Комплектовщик,Рязань,76,1663.7,1628.9,1592.9,1627.4,1628.9,1627.4
10,ordinary,shift,net,Продавец,Москва,71,2344.8,2427.0,2323.9,2331.2,2427.0,2331.2
1,ordinary,shift,net,Грузчик,Санкт-Петербург,62,386.1,370.7,367.4,340.0,370.7,340.0


,scheme,reference,scope,n,groups,scaled_delta,ci_low,ci_high,relative_change_pct
0,by_mode,global,all,3160,1443,0.034,0.004,0.068,4.145
3,by_mode,global,watch,651,343,0.155,0.006,0.325,5.376
5,by_mode,global,remote_explicit,72,68,0.069,0.021,0.130,17.837
6,by_mode,global,watch_shift_le_50000,548,277,-0.036,-0.092,0.010,-11.317
7,by_segment,global,all,1250,515,0.009,-0.000,0.018,5.238
10,by_segment,global,watch,76,50,-0.000,-0.015,0.016,-0.074
11,by_segment,global,watch_shift_le_50000,75,49,-0.001,-0.016,0.017,-0.674
12,hybrid,global,all,3160,1443,0.004,-0.000,0.007,0.451
15,hybrid,global,watch,651,343,-0.000,-0.002,0.002,-0.001
17,hybrid,global,remote_explicit,72,68,0.000,0.000,0.000,0.000


,mode,id,y,p50,abs_error_p50
1,watch,7694158520,1585000.0,12961.5,1572038.5
3,watch,7743890062,510000.0,7158.1,502841.9
6,watch,8169444746,423500.0,6086.9,417413.1
29,ordinary,7883207058,315000.0,4791.0,310209.0
34,ordinary,3753773954,235000.0,5632.3,229367.7
41,ordinary,7231222956,230000.0,23685.5,206314.5
78,remote,8485749984,105000.0,4140.6,100859.4
84,remote,8364559509,70100.0,4069.0,66031.0
92,remote,2459477803,52500.0,3025.6,49474.4


## Проверить новую карту на сохранённых тестовых прогнозах

Эта проверка применяет отредактированные правила ко всем 3 170 размеченным
вакансиям через ранее рассчитанные прогнозы соответствующих тестовых фолдов.
Так можно снова измерить ошибку и охват при смене правила без обучения.
Для нового пакета сначала нужны его прогнозы на общем разбиении.
Результаты после просмотра прежнего теста остаются ретроспективными.

In [6]:
from run_salary_routing import quality_metrics

oof = read('predictions_oof')
replayed = routing.replay_policy(oof, manual, catalog)
combined = pd.concat([oof, replayed], ignore_index=True)
new_quality, new_native = quality_metrics(combined)
checked = new_quality.query("sample == 'available' and scope == 'all' and scheme in ['global','manual','replayed_policy']").copy()
checked['eligible_coverage'] = checked['n'] / checked['available_n']
display(checked[['scheme','n','available_n','eligible_coverage','scaled_pinball','coverage']].round(5))

,scheme,n,available_n,eligible_coverage,scaled_pinball,coverage
0,global,3160,3170,0.99685,0.80983,0.47658
88,manual,3160,3170,0.99685,0.81018,0.47722
107,replayed_policy,3160,3170,0.99685,0.81004,0.47785


## Артефакты

[Таблица качества и охвата](experiments/2026-11-18_salary_routing/summary.csv) ·
[Дерево экспериментов](experiments/2026-11-18_salary_routing/experiment_tree.md) ·
[Редактируемые правила](experiments/2026-11-18_salary_routing/routing_policy.json) ·
[Каталог готовых пакетов](models/2026-11-18_salary_routing/catalog.json)